# Retrieval baseline

Eight original demo passages and questions check the retrieval plumbing. These are authored diagnostics, not an independent ministry-data evaluation. No private book text is included.

In [1]:
# Load the local tools
import json
import sys
import tempfile
from pathlib import Path

root = Path.cwd()
if not (root / "search.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from search import SearchIndex, build_index
from evaluate import evaluate
temporary = tempfile.TemporaryDirectory()
index_path = Path(temporary.name) / "demo.sqlite"
summary = build_index([root / "examples"], index_path)
print("Demo chunks:", summary["chunks"])
print("Import audit:", summary["audit"])

Demo chunks: 8
Import audit: {'source_rows': 8, 'conflicting_sections_excluded': 0, 'accepted_sections': 8}


In [2]:
# Measure title retrieval on the authored questions
questions = [json.loads(line) for line in (root / "examples/questions.jsonl").read_text().splitlines()]
result = evaluate(index_path, questions, "lexical", 4)
print("Queries:", result["queries"])
print("Hit at 4:", result["hit_at_k"])
print("Mean reciprocal rank:", result["mean_reciprocal_rank"])

Queries: 8
Hit at 4: 1.0
Mean reciprocal rank: 1.0


In [3]:
# Inspect a result instead of relying on the aggregate
hits = SearchIndex(index_path).search("Do citations prove every claim?", limit=3)
for rank, hit in enumerate(hits, 1):
    print(rank, hit["title"])
    print(hit["text"][:180])
print("Unmatched query:", SearchIndex(index_path).search("zyxwvut"))
temporary.cleanup()

1 Checking generated answers
Supply retrieved passages to the language model with numbered citations. The model should say when the passages do not support an answer. Validate that returned citation numbers we
Unmatched query: []


Next compare lexical and hybrid retrieval on the same reviewed questions from a real collection. Track retrieval misses separately from unsupported generated answers. Title-level hits here do not measure passage-level recall or answer quality.